# Data Cleaning & Visualization Project
## Retail Sales Analysis

This project cleans a raw retail-sales dataset and creates visualizations to identify useful sales insights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

In [ ]:
# Load the raw dataset
df = pd.read_csv('raw_retail_sales.csv')
print('Rows and columns:', df.shape)
df.head()

## 1. Data Inspection

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nData types:')
print(df.dtypes)

## 2. Data Cleaning

In [ ]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Unit_Price'] = pd.to_numeric(df['Unit_Price'], errors='coerce')

# Remove duplicate records
df = df.drop_duplicates().copy()

# Treat impossible negative values as missing
df.loc[df['Quantity'] <= 0, 'Quantity'] = np.nan
df.loc[df['Unit_Price'] <= 0, 'Unit_Price'] = np.nan

# Fill missing categorical values with the mode
for col in ['Product', 'Region']:
    df[col] = df[col].fillna(df[col].mode()[0])

# Fill missing numeric values with the median
for col in ['Quantity', 'Unit_Price']:
    df[col] = df[col].fillna(df[col].median())

# Handle outliers using IQR clipping
for col in ['Quantity', 'Unit_Price']:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    df[col] = df[col].clip(lower=lower, upper=upper)

# Recalculate sales after cleaning
df['Sales'] = df['Quantity'] * df['Unit_Price']

print('Remaining missing values:')
print(df.isnull().sum())
print('Rows after cleaning:', len(df))

In [ ]:
# Save the cleaned dataset
df.to_csv('cleaned_retail_sales.csv', index=False)
print('cleaned_retail_sales.csv saved successfully.')

## 3. Visualizations

In [ ]:
product_sales = df.groupby('Product')['Sales'].sum().sort_values(ascending=False)
plt.figure(figsize=(9,5))
product_sales.plot(kind='bar')
plt.title('Total Sales by Product')
plt.xlabel('Product')
plt.ylabel('Sales')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
region_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
plt.figure(figsize=(9,5))
region_sales.plot(kind='bar')
plt.title('Total Sales by Region')
plt.xlabel('Region')
plt.ylabel('Sales')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
monthly_sales = df.groupby(df['Order_Date'].dt.to_period('M'))['Sales'].sum()
plt.figure(figsize=(9,5))
monthly_sales.plot(kind='line', marker='o')
plt.title('Monthly Sales Trend')
plt.xlabel('Month')
plt.ylabel('Sales')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9,5))
plt.hist(df['Sales'], bins=15)
plt.title('Distribution of Sales Values')
plt.xlabel('Sales')
plt.ylabel('Number of Orders')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7,5))
sns.heatmap(df[['Quantity','Unit_Price','Sales']].corr(), annot=True, fmt='.2f')
plt.title('Correlation Between Numeric Variables')
plt.tight_layout()
plt.show()

## 4. Key Findings & Conclusion

- Missing values were handled using mode and median imputation.
- Duplicate records were removed.
- Invalid negative values were treated as missing.
- Extreme numeric values were handled using the IQR method.
- Product, region, monthly trend, distribution, and correlation visualizations were created.

**Conclusion:** The cleaned dataset is more consistent and ready for analysis. The visualizations make important sales patterns easier to understand.